In [ ]:
%matplotlib inline

# Version 11 avril 2026
import os, fnmatch, sys
import glob
import numpy as np
import numpy.ma as ma

import pandas as pd
from pathlib import Path
from jupyprint import jupyprint  
from IPython.display import display, FileLink, FileLinks, HTML
from yaml import safe_load

# matplotlib
from matplotlib import colors
from pylab import *
from IPython.display import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

# clawpack
# Source : http://www.clawpack.org/gallery/_static/apps/notebooks/visclaw/gridtools.html#Extract-1d-transects
from clawpack.visclaw import colormaps, frametools, geoplot, gridtools
from clawpack.visclaw import animation_tools
from clawpack.visclaw.plottools import pcolorcells
from clawpack.geoclaw import dtopotools, topotools, marching_front
from clawpack.geoclaw import util
from clawpack.pyclaw.solution import Solution
from clawpack.pyclaw import solution as solution
from clawpack.amrclaw import region_tools



# latex
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['text.latex.preamble'] = r'\usepackage{libertine}'
plt.rcParams['font.size'] = 12

# répertoires et configuration
current_directory = Path().cwd()
proj_dir      = current_directory.parent
topo_dir      = proj_dir / "Topo"
images_dir    = proj_dir / "Figures"
animation_dir = images_dir / 'Animation'
video_dir     = images_dir / 'Vidéos'
output_dir    = current_directory / "_output"
if not animation_dir.exists():
     animation_dir.mkdir(parents=True, exist_ok=True)
     print(f"Création du répertoire {animation_dir}")
if not video_dir.exists():
     video_dir.mkdir(parents=True, exist_ok=True)
     print(f"Création du répertoire {video_dir}")
HOME       = current_directory
répertoire_travail  = HOME 
répertoire_résultat = répertoire_travail / '_output'
#os.path.join(répertoire_travail, '_output')
configuration_file  = proj_dir / "impulse_configuration.yaml"
sys.path.insert(0,répertoire_travail) 
if configuration_file.exists():
    print(f"Chargement du fichier de configuration existant")
    with open(configuration_file) as file:
        config      = safe_load(file)
        topo        = config["topo_files"]
        lake        = config["lake"]
        gauges      = config['gauges']
        rheology    = config['rheology']
        computation = config['computation']
        output      = config['output']
        topo_files  = config['topo_files']
else:
    print(f"Le fichier {configuration_file} est manquant. Il faut le générer avec Waves_preprocess")
print('Répertoire où sont les données : \n  %s' % répertoire_résultat)
if not os.path.isdir(répertoire_travail):
      print('*** Revoir le nom du répertoire (je ne le trouve pas)')

# AVAC
sys.path.insert(0, str(proj_dir / "AVAC"))
from module_avac import reading_raster_file_features, reload_avac, reading_raster_file, \
    export_claw_dem, export_claw_dem_window, determine_file_type, plot_topo, make_output
from module_avac import fn_eta, fn_ground, fn_h, fn_hu, fn_husquare, fn_hv, fn_u, fn_v, fn_extract
from module_avac import format_m
      
# variables
CLAW           = os.environ['CLAW']
format_fichier = output['output_format']
rho            = rheology['rho']
g              = rheology['gravity']
z_lac          = lake['water_level'] if configuration_file.exists() else 1500
nb_grille      = 400 # nombre d'éléments de la grille d'interpolation

topo_fine = reading_raster_file(str(topo_dir / topo_files['fine']))
xmin_mnt, xmax_mnt, ymin_mnt, ymax_mnt, nbx_mnt, nby_mnt, cell_size_mnt, dictionary_mnt, _, _,_ = \
        reading_raster_file_features(topo_dir / lake['topography'])


# Topographie

In [ ]:
# Ensemble des données topo
topo_file = reading_raster_file(topo_dir / lake['topography'])

fig, ax, x0, y0 = plot_topo(topo_file,step=500)
ax.add_patch(plt.Rectangle((lake['xmin']-x0,lake['ymin']-y0) , width=lake['xmax']-lake['xmin'], 
                           height=lake['ymax']-lake['ymin'], ls="-", lw=2, ec="red", fc="none"))




In [ ]:
# Ensemble des données topo et zoom sur le lac

fig, ax, x0, y0 = plot_topo(topo_file, step=50)
ax.add_patch(plt.Rectangle((lake['xmin']-x0, lake['ymin']-y0),
                            width=lake['xmax']-lake['xmin'],
                            height=lake['ymax']-lake['ymin'],
                            ls="-", lw=2, ec="red", fc="none"))

# Zoom sur le lac avec une marge
margin = 100  # mètres
text_m = 10
ax.set_xlim(lake['xmin'] - x0 - margin, lake['xmax'] - x0 + margin)
ax.set_ylim(lake['ymin'] - y0 - margin, lake['ymax'] - y0 + margin)
if gauges['gauge_recording']:
    for k in range(len(gauges)-1):
        x = gauges[str(k)]['x']
        y = gauges[str(k)]['y']
        ax.scatter(x-x0,y-y0,c = 'red',marker='o',s=12)
        ax.text(x-x0,y-y0+text_m,str(1+k),color = 'red')

fig.savefig(images_dir / "vue_domaine_calcul.png",dpi = 300, bbox_inches = 'tight')

In [ ]:
# Extension du MNT
print(f"Domaine du MNT :")
print(f"- xmin = {format_m(xmin_mnt)} m et xmax = {format_m(xmax_mnt)}")
print(f"- ymin = {format_m(ymin_mnt)} m et ymax = {format_m(ymax_mnt)}")

# Calcul

In [ ]:
if configuration_file.exists():
    print(f"Chargement du fichier de configuration existant")
    with open(configuration_file) as file:
        config      = safe_load(file)
        topo        = config["topo_files"]
        lake        = config["lake"]
        gauges      = config['gauges']
        rheology    = config['rheology']
        computation = config['computation']
        output      = config['output']
        topo_files  = config['topo_files']
config

In [ ]:
# Running geoclaw
make_output(config)

In [ ]:
#reading the mass file
plt.close('all')  # Ferme les figures widget avant de repasser en inline
# Plot with contour lines
%matplotlib inline
mass_evolution = pd.read_csv(output_dir/'mass.txt')

text_legend = {'French':r'Masse de fluide (en t)',
               'English':r'Fluid mass (t)'}[config['output']['Language']]
 

fig, ax = plt.subplots(figsize=(8,4))
#ax.plot(mass_evolution['t_s'], mass_evolution['wet_mass_kg'],    label='masse dans le domaine')
ax.plot(mass_evolution['t_s'], mass_evolution['wet_mass_kg']/1e6 )
ax.set_xlabel('$t$ (s)')

ax.set_ylabel(rf'{text_legend}')
ax.legend()
ax.grid()
fig.savefig(images_dir / "evolution_total_mass_lake.png",dpi=300)

# Importation des résultats

In [ ]:
fichiers = fnmatch.filter(os.listdir(répertoire_résultat), 'fort.q*' )
NbSim    = len(fichiers)
print(f"Il y a {NbSim} fichiers fort.q*.")

In [ ]:
frames = []
for i in range(NbSim):
      sol = solution.Solution(i, path=répertoire_résultat, file_format=format_fichier)
      frames.append(sol)

nombre_patches = len(frames[0].domain.patches)

t_0 = frames[0].t
t_f = frames[NbSim-1].t
dt  = (t_f-t_0)/(NbSim-1)

def check_output(Frames):
      xlower = []
      xupper = []
      ylower = []
      yupper = []
      for patch in Frames[0].domain.patches:
            xlower.append(patch.grid.x.lower)
            xupper.append(patch.grid.x.upper)
            ylower.append(patch.grid.y.lower)
            yupper.append(patch.grid.y.upper)
      xmin_o  = min(xlower)
      xmax_o  = max(xupper)
      ymin_o  = min(ylower)
      ymax_o  = max(yupper)
      delta = Frames[0].patch.delta
      print(f"* xmin =  {format_m(xmin_o)} et xmax =  {format_m(xmax_o)} ; delta_x = {delta[0]}.")
      print(f"* ymin =  {format_m(ymin_o)} et ymax = {format_m(ymax_o)} ; delta_y = {delta[1]}.")
      print(f"* Il y a {nombre_patches} patches au temps t = {t_0}.")
      print(f"* Temps initial t = {t_0} ; temps final t = {t_f} ; dt = {dt}.")
      print(f"* Nombre de simulations : {NbSim-1}.")
      domain_extent = {'xmin':xmin_o,'xmax':xmax_o,'ymin':ymin_o,'ymax':ymax_o}
      return domain_extent
computation_domain = check_output(frames)


In [ ]:
xmin, xmax = lake['xmin'], lake['xmax']
ymin, ymax = lake['ymin'], lake['ymax']

# grille
x = np.linspace(xmin,xmax,nb_grille)
y = np.linspace(ymin,ymax,nb_grille)
X_grille, Y_grille = meshgrid(x,y)
dx = (xmax-xmin)/nb_grille
dy = (ymax-ymin)/nb_grille

# interpolation de la solution 
sol_0     = gridtools.grid_output_2d(frames[0], fn_ground, X_grille, Y_grille, 
                                    levels='all',return_ma=True)
eta_0     = gridtools.grid_output_2d(frames[0], fn_eta, X_grille, Y_grille, 
                                      levels='all',return_ma=True)
hauteur_0 = gridtools.grid_output_2d(frames[0], fn_h, X_grille, Y_grille, 
                                      levels='all',return_ma=True)

volume_0 = int(hauteur_0.sum()*dx*dy)
print("La profondeur du lac est {:.2f} m.".format(hauteur_0.max()))
print(f"Le volume du lac est {format_m(volume_0)} m³." )

zmin = eta_0.min()
zmax = eta_0.max()
print(f"Cote minimale du MNT : {zmin:.1f} m. Cote maximale du MNT : {zmax:.1f} m.")
print(f"Cote maximale : {lake['water_level']} m.")



In [ ]:
# Tracé de la condition initiale
fig, ax, x0, y0 = plot_topo(topo_file, step=50)
ax.add_patch(plt.Rectangle((lake['xmin']-x0, lake['ymin']-y0),
                            width=lake['xmax']-lake['xmin'],
                            height=lake['ymax']-lake['ymin'],
                            ls="-", lw=1, ec="red", fc="none"))

# Zoom sur le lac avec une marge
margin = 100  # mètres
 
ax.set_xlim(lake['xmin'] - x0 - margin, lake['xmax'] - x0 + margin)
ax.set_ylim(lake['ymin'] - y0 - margin, lake['ymax'] - y0 + margin)


eta_masque = ma.masked_where(hauteur_0 < computation['dry_limit'], hauteur_0)
cmap_flat = ListedColormap([
                            (0,   0,   0,   0  ), # transparent
                            (0.2, 0.4, 1.0, 0.95) ]) # bleu])   
ax.imshow((hauteur_0 >= computation['dry_limit']).astype(float), origin='lower',
          extent=[xmin-x0, xmax-x0, ymin-y0, ymax-y0],
          cmap=cmap_flat, vmin=0, vmax=1)

 

#  Vérification des conditions aux limites

In [ ]:
def valeurs_frontière(face,i,origine = True):
      if face == 'sud':
            x_profil = np.linspace(xmin, xmax, nb_grille)
            y_profil = ymin*np.ones(x_profil.shape)
            h_out = gridtools.grid_output_2d(frames[i], fn_h, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            hv_out = gridtools.grid_output_2d(frames[i], fn_hv, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            eta_out = gridtools.grid_output_2d(frames[i], fn_eta, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            v_out  = np.zeros(x_profil.shape)
            np.divide(hv_out, h_out, out = v_out, where = hv_out!=0)
            if origine == False: x_profil = x_profil-xmin
            return x_profil, h_out, v_out, hv_out, eta_out
      if face == 'nord':
            x_profil = np.linspace(xmin, xmax, nb_grille)
            y_profil = ymax*np.ones(x_profil.shape)
            h_out = gridtools.grid_output_2d(frames[i], fn_h, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            hv_out = gridtools.grid_output_2d(frames[i], fn_hv, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            eta_out = gridtools.grid_output_2d(frames[i], fn_eta, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            v_out  = np.zeros(x_profil.shape)
            np.divide(hv_out, h_out, out = v_out, where = hv_out!=0)
            if origine == False: x_profil = x_profil-xmin
            return x_profil, h_out, v_out, hv_out, eta_out
      if face == 'est':
            y_profil = np.linspace(ymin, ymax, nb_grille)
            x_profil = xmax*np.ones(y_profil.shape)
            h_out = gridtools.grid_output_2d(frames[i], fn_h, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            hv_out = gridtools.grid_output_2d(frames[i], fn_hu, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            eta_out = gridtools.grid_output_2d(frames[i], fn_eta, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            v_out  = np.zeros(x_profil.shape)
            np.divide(hv_out, h_out, out = v_out, where = hv_out!=0)
            if origine == False: y_profil = y_profil-ymin
            return y_profil, h_out, v_out, hv_out, eta_out
      if face == 'ouest':
            y_profil = np.linspace(ymin, ymax, nb_grille)
            x_profil = xmin*np.ones(y_profil.shape)
            h_out = gridtools.grid_output_2d(frames[i], fn_h, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            hv_out = gridtools.grid_output_2d(frames[i], fn_hu, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            eta_out = gridtools.grid_output_2d(frames[i], fn_eta, x_profil, y_profil, 
                                 levels='all',return_ma=True)
            v_out  = np.zeros(x_profil.shape)
            np.divide(hv_out, h_out, out = v_out, where = hv_out!=0)
            if origine == False: y_profil = y_profil-ymin
            return y_profil, h_out, v_out, hv_out, eta_out



In [ ]:
temps_in = np.arange(t_0,t_f+dt,dt)
versant = 'sud'
L = 93
rapport_L = 93/(ymax-ymin)
loc_légende = 'upper right'
fig, ((ax1, ax2,ax3)) = plt.subplots(1,3)
fig.set_figheight(4)
fig.set_figwidth(14)
## fig 1 ##
flux_masse_versant = -1*np.array([valeurs_frontière(versant,i)[3].mean() for i in range(0,NbSim)])
#df = (pd.read_csv(répertoire_travail+'/CL/'+'hv_'+versant+'.dat',sep = '\t',names = ['temps','hv'])).drop(range(0)  )
#df['temps']=df['temps']-7
ax1.plot(temps_in, flux_masse_versant,label='CL')
ax1.set_xlabel(r"$t$ [s]")
ax1.set_ylabel(r"$\dot M$ [m$^2$/s]")

#ax1.plot(df['temps'],df['hv']*rapport_L,label='sortie AVAC')
ax1.legend(loc=loc_légende,ncol=2)
## fig 2 ##
hauteur_versant = np.array([valeurs_frontière(versant,i)[1].mean() for i in range(0,NbSim)])
# df=(pd.read_csv(répertoire_travail+'/CL/'+'hauteur_'+versant+'.dat',sep = '\t',names = ['temps','h'])).drop(range(0) )
# df['temps']=df['temps']-7
ax2.plot(temps_in, hauteur_versant,label='CL')
ax2.set_xlabel(r"$t$ [s]")
ax2.set_ylabel(r"$h$ [m]")

#ax2.plot(df['temps'],df['h']*rapport_L,label='sortie AVAC')
ax2.legend(loc=loc_légende,ncol=2)
## fig 3 ##
vitesse_versant = -1*np.array([valeurs_frontière(versant,i)[2].mean() for i in range(0,NbSim)])
# df=(pd.read_csv(répertoire_travail+'/CL/'+'vitesse_'+versant+'.dat',sep = '\t',names = ['temps','v'])).drop(range(0) )
# df['temps']=df['temps']-7
ax3.plot(temps_in, vitesse_versant,label='CL')
ax3.set_xlabel(r"$t$ [s]")
ax3.set_ylabel(r"$v$ [m/s]")

#ax3.plot(df['temps'],df['v']*rapport_L,label='sortie AVAC')
ax3.legend(loc=loc_légende,ncol=2)

 

# Profil

In [ ]:
temps_in = np.arange(t_0,t_f+dt,dt)
# grille
x_début = 965840
y_début = 6536175
x_fin   = 965730
y_fin   = 6536260

x_profil = np.linspace(x_début, x_fin, nb_grille)
y_profil = np.linspace(y_début, y_fin, nb_grille)
distance = ((x_profil-x_début)**2+(y_profil-y_début)**2)**0.5
distance.shape

In [ ]:


# unpack the results:
ground_1d = gridtools.grid_output_2d(frames[0], fn_ground, x_profil, y_profil,
                                     levels='all', return_ma=True, method='linear')
eta_1d    = gridtools.grid_output_2d(frames[0], fn_eta,    x_profil, y_profil,
                                     levels='all', return_ma=True, method='linear')
z_coupe_min = eta_1d.min()-5

print(f"solution au temps {frames[0].t} s")
fig = plt.figure(figsize=(10, 4))
axes = plt.subplot(1, 1, 1)
axes.set_ylim([z_coupe_min,eta_1d.max()+1])
axes.fill_between(distance, ground_1d, z_coupe_min,
                  hatch='//', edgecolor='sienna', facecolor='white')
axes.plot(distance, ground_1d, color='sienna')
axes.fill_between(distance, eta_1d, ground_1d,
                  where=(eta_1d > ground_1d), color='skyblue')
axes.plot(distance, ma.masked_where(eta_1d == ground_1d, eta_1d), color='deepskyblue')



In [ ]:

z_profil_min = sol_0.min()-0.5
z_profil_min = z_lac-9
z_profil_max = sol_0.max()-1
z_profil_max = z_lac +6


figs = []
def ExportAnimation():
    x1 = distance[-1] 
    x0 = distance[0]
    L = x1-x0
    for i in range(0,NbSim):
        eta_i = gridtools.grid_output_2d(frames[i], fn_eta, x_profil, y_profil, 
                                 levels='all',return_ma=True,method='linear')
        sol_i = gridtools.grid_output_2d(frames[i], fn_ground, x_profil, y_profil, 
                                 levels='all',return_ma=True,method='linear')
        h_i = gridtools.grid_output_2d(frames[i], fn_h, x_profil, y_profil, 
                                 levels='all',return_ma=True,method='linear')
        fig  = plt.figure( figsize=(10,2))
        axes = plt.subplot(1, 1, 1)
        
        axes.set_xlabel(r'$x $',fontsize=14)
        axes.set_ylabel(r'$z $',fontsize=14)
        axes.set_xlim((0,L))
        axes.set_ylim(( z_profil_min, z_profil_max))

        text = axes.text(L/2, z_profil_max-1.5, '')
        tt = temps_in[i]
        val = f'{tt:.1f}'
        text.set_text(r'$ t = {} $ s '.format(val))

        axes.set_title(" ")
        axes.fill_between(distance, eta_i, sol_i, color='lightskyblue')
        axes.fill_between(distance, sol_i, z_profil_min,  hatch='//', edgecolor='sienna',facecolor='white')

        axes.plot(distance, ma.masked_where(eta_i==sol_i, eta_i), color = 'deepskyblue')
        #axes.plot(distance, eta_i, 'b')
        figs.append(fig)
        plt.close(fig)
    return figs


figures = ExportAnimation()
animation_tools.interact_animate_figs(figures, manual=True)

In [ ]:


# Animation
images = animation_tools.make_images(figures)
anim = animation_tools.animate_images(images, figsize=(12,4))
HTML(anim.to_jshtml())

In [ ]:

for i in range(len(figs)):
    figs[i].savefig(str(animation_dir / 'profil_vague')+str(i)+'.png', bbox_inches='tight',dpi=300)

In [ ]:
# export
file_name = 'profil-vagues.mp4'
animation_tools.make_mp4( anim, file_name=video_dir /file_name)

# Énergie totale et flux

## énergie du lac

In [ ]:
énergie_potentielle = np.zeros((NbSim,)+hauteur_0.shape)
vague               = np.zeros((NbSim,)+hauteur_0.shape)
énergie_cinétique   = np.zeros((NbSim,)+hauteur_0.shape)
hauteur             = np.zeros((NbSim,)+hauteur_0.shape)
from tqdm import tqdm
for i in tqdm(range(NbSim)):
      hauteur[i] = gridtools.grid_output_2d(frames[i], fn_h, X_grille, Y_grille, 
                                    levels='all',return_ma=True) 
      vague[i]   = hauteur[i]- hauteur_0
      énergie_potentielle[i] = 0.5*rho*g*vague[i]**2
      énergie_cinétique[i]   = 0.5*rho*(gridtools.grid_output_2d(frames[i], fn_hu, X_grille, Y_grille, 
                                    levels='all',return_ma=True)**2 + \
                                          gridtools.grid_output_2d(frames[i], fn_hv, X_grille, Y_grille, 
                                    levels='all',return_ma=True)**2)
      np.divide(énergie_cinétique[i], hauteur[i], out = énergie_cinétique[i], where = hauteur[i]!=0)

In [ ]:
énergie_pot_lac = np.array([énergie_potentielle[i].sum()*dx*dy/1e9 for i in range(NbSim)]) 
énergie_cin_lac = np.array([énergie_cinétique[i].sum()*dx*dy/1e9 for i in range(NbSim)]) 
énergie_tot_lac = énergie_cin_lac+énergie_pot_lac 

In [ ]:

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(temps_in, énergie_tot_lac,label=r'$E$')
ax.plot(temps_in, énergie_cin_lac,'--',label=r'$E_c$')
plt.grid(axis="y", color = 'gray', alpha = 0.25)
plt.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_ylabel(r'énergie   [GJ]')
ax.set_xlabel(r'temps $t$ [s]')  
fig.legend(loc="upper left",ncol=2)

## Hauteur et volume dans le lac

In [ ]:
variation_hauteur = np.array([hauteur[i].sum()*dx*dy  for i in range(NbSim)])-volume_0

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(temps_in, variation_hauteur,label=r'$E$')

ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_ylabel(r'$\Delta V$   [m$^3$]')
ax.set_xlabel(r'temps $t$ [s]')  
#fig.legend(loc="upper left",ncol=2)
jupyprint(f"L'augmentation de volume est de {format_m(variation_hauteur[-1],0)}"+" $\\text{m}^3 $.")

## Flux de masse

In [ ]:

fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(10)
fig.set_figwidth(10)

## fig 1 ##
flux_masse  = {'est'   : np.array([valeurs_frontière('est',i)[3].mean() for i in range(0,NbSim)]),
     'sud'  : np.array([valeurs_frontière('sud',i)[3].mean() for i in range(0,NbSim)]),
     'nord'  : np.array([valeurs_frontière('nord',i)[3].mean() for i in range(0,NbSim)]),
     'ouest' : np.array([valeurs_frontière('ouest',i)[3].mean() for i in range(0,NbSim)]),
     }

dict_boundary = {'sud':'(a) sud', 'est':'(b) est', 'nord':'(c) nord', 'ouest':'(d) ouest'}



boundaries = ('est','ouest','sud','nord')
axes_list =  (ax1,ax2,ax3,ax4)
for b, ax in zip(boundaries,axes_list):
    ax.plot(temps_in,flux_masse[b])
    ax.grid()
    ax.text(0.9, 0.9, dict_boundary[b],
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes)
for ax in (ax3,ax4):
    ax.set_xlabel(r"$t$ (s)")
for ax in (ax1,ax3):
    ax.set_ylabel(r"$Q$ (m³/s)")

fig.savefig(images_dir / "flux_masses_vagues.png",dpi = 300, bbox_inches = 'tight')


In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
flux_masse_total = (xmax-xmin)*(flux_masse['sud']-flux_masse['nord'])+(ymax-ymin)*(-flux_masse['est']+flux_masse['ouest'])
ax.plot(temps_in, flux_masse_total,label=r'$E$')

ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_ylabel(r'Flux total   [m$^3$/s]')
ax.set_xlabel(r'temps $t$ [s]')  
#fig.legend(loc="upper left",ncol=2)

jupyprint(f"L'augmentation de volume est de {format_m(flux_masse_total.sum())}"+" $\\text{m}^3 $.")

## Flux d'énergie cinétique et d'énergie potentielle

### Énergie cinétique

In [ ]:
fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(10)
fig.set_figwidth(10)

## fig 1 ##
flux_énergie  = {
    'est'   : np.array([(valeurs_frontière('est',i)[3]*énergie_cinétique[i][:,-1]).mean() for i in range(0,NbSim)]) ,
     'sud'  : np.array([(valeurs_frontière('sud',i)[3]*énergie_cinétique[i][0,:]).mean() for i in range(0,NbSim)]),
     'nord' : np.array([(valeurs_frontière('nord',i)[3]*énergie_cinétique[i][-1,:]).mean() for i in range(0,NbSim)]),
     'ouest': np.array([(valeurs_frontière('ouest',i)[3]*énergie_cinétique[i][:,0]).mean() for i in range(0,NbSim)])
     }

dict_boundary = {'sud':'(a) sud', 'est':'(b) est', 'nord':'(c) nord', 'ouest':'(d) ouest'}



boundaries = ('est','ouest','sud','nord')
axes_list =  (ax1,ax2,ax3,ax4)
for b, ax in zip(boundaries,axes_list):
    ax.plot(temps_in,flux_énergie[b]/1e3)
    ax.grid()
    ax.text(0.9, 0.9, dict_boundary[b],
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes)
for ax in (ax3,ax4):
    ax.set_xlabel(r"$t$ (s)")
for ax in (ax1,ax3):
    ax.set_ylabel(r"$\dot E_c$ [kW/m]")

fig.savefig(images_dir / "flux_énergie_vagues.png",dpi = 300, bbox_inches = 'tight')

In [ ]:
fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(10)
fig.set_figwidth(10)

## fig 1 ##
flux_potentielle  = {
    'est'   : rho*g*np.array([(valeurs_frontière('est',i)[3]* \
                                  (sol_0[:,-1]+hauteur[i][:,-1]-z_lac)).mean() for i in range(0,NbSim)]) ,
     'sud'  : rho*g*np.array([(valeurs_frontière('sud',i)[3]* \
                                  (sol_0[0,:]+hauteur[i][0,:]-z_lac)).mean() for i in range(0,NbSim)]) ,
     'nord' : rho*g*np.array([(valeurs_frontière('nord',i)[3]* \
                                    (sol_0[-1,:]+hauteur[i][-1,:]-z_lac)).mean() for i in range(0,NbSim)]) ,
     'ouest': rho*g*np.array([(valeurs_frontière('ouest',i)[3]* \
                                    (sol_0[:,0]+hauteur[i][:,0]-z_lac)).mean() for i in range(0,NbSim)])
     }

dict_boundary = {'sud':'(a) sud', 'est':'(b) est', 'nord':'(c) nord', 'ouest':'(d) ouest'}



boundaries = ('est','ouest','sud','nord')
axes_list =  (ax1,ax2,ax3,ax4)
for b, ax in zip(boundaries,axes_list):
    ax.plot(temps_in,flux_potentielle[b]/1e6)
    ax.grid()
    ax.text(0.9, 0.9, dict_boundary[b],
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes)
for ax in (ax3,ax4):
    ax.set_xlabel(r"$t$ (s)")
for ax in (ax1,ax3):
    ax.set_ylabel(r"$\dot E_p $ [MW/m]")

fig.savefig(images_dir / "flux_énergie_potentielle_vagues.png",dpi = 300, bbox_inches = 'tight')

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
flux_cinétique_total = (xmax-xmin)*(flux_énergie['sud']-flux_énergie['nord'])+(ymax-ymin)*(-flux_énergie['est']+flux_énergie['ouest'])
flux_potentielle_total = (xmax-xmin)*(flux_potentielle['sud']-flux_potentielle['nord'])+(ymax-ymin)*(-flux_potentielle['est']+flux_potentielle['ouest'])
flux_énergie_total   = flux_cinétique_total + flux_potentielle_total
ax.plot(temps_in, flux_énergie_total/1e6,label=r'$E$')

plt.grid(axis="y", color = 'gray', alpha = 0.25)
plt.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_ylabel(r"Flux d'énergie   [GW]")
ax.set_xlabel(r'temps $t$ [s]')  
#fig.legend(loc="upper left",ncol=2)


In [ ]:
from itertools import accumulate
from operator import add 
 
énergie_fournie = np.array(list(accumulate(flux_énergie_total, add))) 
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(temps_in, énergie_fournie/1e6,label=r'$E$')

plt.grid(axis="y", color = 'gray', alpha = 0.25)
plt.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_ylabel(r"Énergie fournie   [GJ]")
ax.set_xlabel(r'temps $t$ [s]') 

In [ ]:
énergie_fournie = np.array(list(accumulate(flux_énergie_total, add))) 
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(énergie_fournie/1e9,énergie_tot_lac,label=r'$E$')

plt.grid(axis="y", color = 'gray', alpha = 0.25)
plt.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_xlabel(r"Énergie fournie   [GJ]")
ax.set_ylabel(r"Énergie totale du lac   [GJ]") 

In [ ]:
fig.savefig(images_dir / "efficience_énergie.png", bbox_inches = 'tight')

In [ ]:
énergie_fournie = np.array(list(accumulate(flux_énergie_total, add))) 
efficience = np.divide(énergie_tot_lac * 1e9, 
                       énergie_fournie, 
                       where=(énergie_fournie > 1e-3),
                       out=np.full_like(énergie_fournie, np.nan, dtype=float))
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(temps_in,efficience,label=r'$E$')

plt.grid(axis="y", color = 'gray', alpha = 0.25)
plt.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_xlabel(r"$t$   [s]")
ax.set_ylabel(r"$E_{lac}/E_{aval.}$ ") 
print(f"max efficience = {np.nanmax(efficience[efficience<1])  }")

# Jauges

In [ ]:
column_name =['level', 'time', 'h', 'hu', 'hv', 'eta' ]
fichier = 'gauge00001.txt'
data1 = pd.read_csv(répertoire_résultat / fichier, sep='\\s+',  comment='#', encoding='latin-1',names =  column_name, index_col=False)
fichier = 'gauge00002.txt'
data2 = pd.read_csv(répertoire_résultat / fichier, sep='\\s+',  comment='#', encoding='latin-1',names =  column_name, index_col=False)
fichier = 'gauge00003.txt'
data3 = pd.read_csv(répertoire_résultat / fichier, sep='\\s+',  comment='#', encoding='latin-1',names =  column_name, index_col=False)
fichier = 'gauge00004.txt'
data4 = pd.read_csv(répertoire_résultat / fichier, sep='\\s+',  comment='#', encoding='latin-1',names =  column_name, index_col=False)

étiquette = ['(1)','(2)','(3)','(4)']
 
def tracer_jauge(ax,data,k):
    ax.plot(data['time'], data['h'],label='CL')
    ax.set_xlabel(r"$t$ [s]")
    ax.set_ylabel(r"$h$ [m]")
    ax.grid()
    plt.text(0.9, 0.9, étiquette[k],
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes)
    
fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(9)
fig.set_figwidth(10)
tracer_jauge(ax1,data1,0)
tracer_jauge(ax2,data2,1)
tracer_jauge(ax3,data3,2)
tracer_jauge(ax4,data4,3)


In [ ]:
fig.savefig(images_dir / 'évolution_jauges.png',dpi=300, bbox_inches='tight')

In [ ]:
positions_jauges = []
if gauges['gauge_recording']:
        for k in range(len(gauges)-1):
            gaugeno = str(k)
            x = gauges[gaugeno]['x']
            y = gauges[gaugeno]['y']
            print(f"x = {x} y = {y}")
            positions_jauges.append([x,y])

positions_jauges

In [ ]:
from shapely import LineString, intersection

surface_libre_simple = LineString([(-55, z_lac), (55, z_lac)])

positions_jauges = np.array(positions_jauges)

xcenter, ycenter = [X_grille[hauteur_0>0].mean(), Y_grille[hauteur_0>0].mean()]

xcenter, ycenter = 965800, 6536225-50

étiquette = ['(a)','(b)','(c)','(d)']
étiquette = ['(1)','(2)','(3)','(4)']

def Enlever_masque(fd):
    fdd = []
    for m in range(len(fd)):
        if type(fd[m][1]) != np.ma.core.MaskedConstant:
            fdd.append(fd[m])
    return fdd

def triangle(axes,x_t, z, taille, rapport):
    cote = taille/np.tan(60/180*np.pi)*rapport
    dh = taille /3
    lw = 0.5
    axes.plot([x_t, x_t+cote ], [z, z+taille], color='b', linestyle='-', linewidth=lw)
    axes.plot([x_t, x_t-cote ], [z, z+taille], color='b', linestyle='-', linewidth=lw)
    axes.plot([x_t+cote, x_t-cote], [z+taille, z+taille], color='b', linestyle='-', linewidth=lw)
    axes.plot([x_t+cote, x_t-cote], [z-dh, z-dh], color='b', linestyle='-', linewidth=lw)
    axes.plot([x_t+cote/2, x_t-cote/2], [z-2*dh, z-2*dh], color='b', linestyle='-', linewidth=lw)

def dessiner_profil(k, ax):
    xj = positions_jauges[k][0]
    yj = positions_jauges[k][1]
    vec_directeur = [xj-xcenter, yj-ycenter]
    vec_directeur = vec_directeur / np.linalg.norm(vec_directeur)

    L_p = 50
    x_0 = xj+vec_directeur[0]*L_p
    x_1 = xj-vec_directeur[0]*L_p
    y_0 = yj+vec_directeur[1]*L_p
    y_1 = yj-vec_directeur[1]*L_p

    nb_points = 40
    x_profil = np.linspace(x_0, x_1, nb_points)
    y_profil = np.linspace(y_0, y_1, nb_points)
    distance = np.array([np.dot([x_profil[i]-xj, y_profil[i]-yj], vec_directeur) for i in range(nb_points)])
    z_profil = gridtools.grid_output_2d(frames[0], fn_ground, x_profil, y_profil, levels='all', return_ma=True)

    # Segment mouille le plus bas (= lac) parmi tous les segments sous z_lac
    z_arr = np.ma.filled(z_profil, z_lac + 1)   # valeurs masquees -> sec
    wet = z_arr <= z_lac
    segments, i = [], 0
    while i < len(wet):
        if wet[i]:
            j = i
            while j < len(wet) and wet[j]:
                j += 1
            segments.append((i, j - 1))
            i = j
        else:
            i += 1
    if not segments:
        return
    i0, i1 = min(segments, key=lambda s: z_arr[s[0]:s[1]+1].min())

    # Rives exactes par interpolation lineaire
    x_right = (distance[i0-1] + (z_lac - z_arr[i0-1]) / (z_arr[i0] - z_arr[i0-1])
               * (distance[i0] - distance[i0-1])) if i0 > 0 else distance[i0]
    x_left  = (distance[i1+1] + (z_lac - z_arr[i1+1]) / (z_arr[i1] - z_arr[i1+1])
               * (distance[i1] - distance[i1+1])) if i1 < len(wet)-1 else distance[i1]

    mask = np.zeros(len(distance), dtype=bool)
    mask[i0:i1+1] = True

    x_t = -40

    ax.text(0.1, 0.9, étiquette[k],
        horizontalalignment='center',
        verticalalignment='center',
        transform=ax.transAxes)
    ax.plot(distance, z_profil, color='sienna')
    ax.fill_between(distance, z_lac, z_profil,
                    where=mask, color='lightskyblue', interpolate=True)
    ax.plot([x_left, x_right], [z_lac, z_lac], color='blue', linestyle='-', linewidth=1)
    triangle(ax, x_t, z_lac, 0.8, 4)
    ax.fill_between(distance, z_profil, z_profil.min(), hatch='OO', edgecolor='peachpuff', facecolor='white')


In [ ]:
# Ensemble des données topo
topo_file = reading_raster_file(topo_dir / lake['topography'])

fig, ax, x0, y0 = plot_topo(topo_file, step=50)
ax.add_patch(plt.Rectangle((lake['xmin']-x0, lake['ymin']-y0),
                            width=lake['xmax']-lake['xmin'],
                            height=lake['ymax']-lake['ymin'],
                            ls="-", lw=2, ec="red", fc="none"))

# Zoom sur le lac avec une marge
margin = 100  # mètres
text_m = 10
ax.set_xlim(lake['xmin'] - x0 - margin, lake['xmax'] - x0 + margin)
ax.set_ylim(lake['ymin'] - y0 - margin, lake['ymax'] - y0 + margin)
# if gauges['gauge_recording']:
#     for k in range(len(gauges)-1):
#         x = gauges[str(k)]['x']
#         y = gauges[str(k)]['y']
#         ax.scatter(x-x0,y-y0,c = 'red',marker='o',s=12)
#         ax.text(x-x0,y-y0+text_m,str(1+k),color = 'red')

for k in range(len(positions_jauges)):
    x = positions_jauges[k][0]
    y = positions_jauges[k][1]
    plt.scatter(x-x0,y-y0,c = 'red',marker='o',s=12)
plt.scatter(xcenter-x0,ycenter-y0,c = 'red',marker='+',s=15)
 


In [ ]:
# Profil en travers avec trace du fond et de la surface libre
fig, ax = plt.subplots(figsize=(12,4))
k = 1
xj = positions_jauges[k][0]
yj = positions_jauges[k][1]
vec_directeur = [xj-xcenter, yj-ycenter]
vec_directeur = vec_directeur / np.linalg.norm(vec_directeur)

L_p = 50
x_0 = xj+vec_directeur[0]*L_p
x_1 = xj-vec_directeur[0]*L_p
y_0 = yj+vec_directeur[1]*L_p
y_1 = yj-vec_directeur[1]*L_p

nb_points = 40
x_profil = np.linspace(x_0, x_1, nb_points)
y_profil = np.linspace(y_0, y_1, nb_points)
distance = np.array([np.dot([x_profil[i]-xj, y_profil[i]-yj], vec_directeur) for i in range(nb_points)])
z_profil = gridtools.grid_output_2d(frames[0], fn_ground, x_profil, y_profil, levels='all', return_ma=True)

# Segment mouille le plus bas (= lac) parmi tous les segments sous z_lac
z_arr = np.ma.filled(z_profil, z_lac + 1)
wet = z_arr <= z_lac
segments, i = [], 0
while i < len(wet):
    if wet[i]:
        j = i
        while j < len(wet) and wet[j]:
            j += 1
        segments.append((i, j - 1))
        i = j
    else:
        i += 1

i0, i1 = min(segments, key=lambda s: z_arr[s[0]:s[1]+1].min())

# Rives exactes par interpolation lineaire
x_right = (distance[i0-1] + (z_lac - z_arr[i0-1]) / (z_arr[i0] - z_arr[i0-1])
           * (distance[i0] - distance[i0-1])) if i0 > 0 else distance[i0]
x_left  = (distance[i1+1] + (z_lac - z_arr[i1+1]) / (z_arr[i1] - z_arr[i1+1])
           * (distance[i1] - distance[i1+1])) if i1 < len(wet)-1 else distance[i1]

mask = np.zeros(len(distance), dtype=bool)
mask[i0:i1+1] = True

x_t = -40

ax.text(0.1, 0.9, étiquette[k],
    horizontalalignment='center',
    verticalalignment='center',
    transform=ax.transAxes)
ax.plot(distance, z_profil, color='sienna')
ax.fill_between(distance, z_lac, z_profil,
                where=mask, color='lightskyblue', interpolate=True)
ax.plot([x_left, x_right], [z_lac, z_lac], color='blue', linestyle='-', linewidth=1)
triangle(ax, x_t, z_lac, 0.8, 3)
ax.fill_between(distance, z_profil, z_profil.min(), hatch='OO', edgecolor='peachpuff', facecolor='white')


In [ ]:
# Figure récapitulative
fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(6)
fig.set_figwidth(12)
dessiner_profil(0,ax1)
dessiner_profil(1,ax2)
dessiner_profil(2,ax3)
dessiner_profil(3,ax4)

In [ ]:
fig.savefig(images_dir / 'profils_jauges.png',dpi=300, bbox_inches='tight')

# Flux sortant

In [ ]:
temps_in = np.arange(t_0,t_f+dt,dt)
# grille
x_début = xmin
y_début = ymax
x_fin   = xmax
y_fin   = ymax

x_profil = np.linspace(x_début, x_fin, nb_grille)
y_profil = np.linspace(y_début, y_fin, nb_grille)
distance = ((x_profil-x_début)**2+(y_profil-y_début)**2)**0.5
distance.shape

débit_sortant = []

for i in range(NbSim):
    qout = gridtools.grid_output_2d(frames[i], fn_hu, x_profil, y_profil, 
                                 levels='all',return_ma=True)
    débit_sortant.append(-1*qout.sum()*dx)

 

fig, ax = plt.subplots(figsize=(12,4))
 

plt.grid(axis="y", color = 'gray', alpha = 0.25)
plt.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_xlabel(r"$t$   [s]")
ax.set_ylabel(r"$Q$   [m$^3$/s]") 
ax.plot(temps_in,débit_sortant)

print(f"Qmax = {np.max(débit_sortant):0.1f} m3/s")

In [ ]:
fig.savefig(images_dir / "débit_surverse.png",bbox_inches = 'tight',dpi=300)

In [ ]:
jupyprint(f"surverse = {int(np.array(débit_sortant).sum()*dt)}  m $ ^3$ ")
print(f"zmax = {max([vague[i].max() for i in range(NbSim)] ):.2f}")
print(f"zmin = {min([vague[i].min() for i in range(NbSim)] ):.2f}")

# Cartes

In [ ]:
# Vue rapprochée du lac
#topo_lac = topotools.Topography(str(topo_dir / lake['topography']), topo_type=3)
topo_lac        = reading_raster_file(topo_dir / lake['topography'])
filter_region   = (xmin, xmax, ymin, ymax)
topo_zoom       = topo_lac.crop(filter_region)
fig, ax, x0, y0 = plot_topo(topo_zoom,contour_interval=5,step=20)

In [ ]:
# Tracé de l'état initial
vague_masque              = np.zeros((NbSim,)+hauteur_0.shape)
hauteur_masque            = np.zeros((NbSim,)+hauteur_0.shape)
for i in range(NbSim):
      # vague_masque[i] = ma.masked_where(vague[i] > 0.01, vague[i])
      # hauteur_masque[i] = ma.masked_where(vague[i] > 0.01, hauteur[i])
      vague_masque[i] = vague[i]
      hauteur_masque[i] = hauteur[i]
      hauteur_masque[i][hauteur[i]<computation['dry_limit']] = np.nan
      vague_masque[i][hauteur[i]<computation['dry_limit']  ] = np.nan
      # hauteur_masque[i] = ma.masked_where(vague[i] > 0.01, hauteur[i])



In [ ]:
from matplotlib import animation
from IPython.display import HTML
matplotlib.rcParams['animation.embed_limit'] = 100
def make_wave_animation(frames, vague_masque, topo_zoom, lake, x0, y0,
                        step = 20, contour_interval=5,
                        vmax=0.5, fps=5, figsize=(10, 8)):
    """
    Animation vue de dessus de la vague (eta - eta_0).
    
    Paramètres
    ----------
    frames       : liste des Solution clawpack
    vague_masque : array (NbSim, ny, nx) pré-calculé (valeurs NaN hors eau)
    topo_zoom    : objet retourné par reading_raster_file (topo recadrée sur le lac)
    lake         : dict avec xmin, xmax, ymin, ymax, water_level
    x0, y0       : coin SW de topo_zoom (retournés par plot_topo)
    vmax         : amplitude max de la colorbar (symétrique ±vmax)
    fps          : images par seconde
    figsize      : taille de la figure
    """
    xmin, xmax = lake['xmin'], lake['xmax']
    ymin, ymax = lake['ymin'], lake['ymax']
    vmax = max([vague[i].max() for i in range(NbSim)] )
    vmin = min([vague[i].min() for i in range(NbSim)] )

    # --- figure de base (fond topo) ---
    fig, ax, x0, y0 = plot_topo(topo_zoom, contour_interval= contour_interval, step=step,
                                 figsize_width=figsize[0])
    ax.set_xlim(xmin - x0  , xmax - x0  )
    ax.set_ylim(ymin - y0  , ymax - y0  )

    # --- artiste imshow (initialisé sur la première frame) ---
    # im = ax.imshow(vague_masque[0], origin='lower',
    #                extent=[xmin - x0, xmax - x0, ymin - y0, ymax - y0],
    #                cmap='viridis',vmin=-vmax, vmax=vmax, alpha=0.95) #,
    # fond fixe : lac initial en bleu
    cmap_lac = ListedColormap([(0, 0, 0, 0),(0.5, 0.7, 1.0, 0.8)])
    ax.imshow((hauteur_masque[0] > 0).astype(float), origin='lower',
            extent=[xmin-x0, xmax-x0, ymin-y0, ymax-y0],
            cmap=cmap_lac, vmin=0, vmax=1, zorder=2)

    im = ax.imshow(vague_masque[0], origin='lower',
               extent=[xmin-x0, xmax-x0, ymin-y0, ymax-y0],
               cmap='coolwarm', vmin=-vmax, vmax=vmax, alpha=0.85, zorder=3)



    fig.colorbar(im, ax=ax, shrink=0.6, label=r'$\eta - \eta_0$ (m)')

    title_text = ax.set_title(rf"$t = {frames[0].t:.1f}$ s")

    # --- fonction de mise à jour ---
    def update(i):
        im.set_data(vague_masque[i])
        # par-dessus : vague animée
        
        title_text.set_text(rf"$t = {frames[i].t:.1f}$ s")
        return im, title_text

    anim = animation.FuncAnimation(fig, update, frames=len(frames),
                                   interval=1000 // fps, blit=True)
    plt.close(fig)   # évite l'affichage statique en plus
    return anim

anim = make_wave_animation(frames, vague_masque, topo_zoom, lake, x0, y0 )
HTML(anim.to_jshtml())


In [ ]:
animation_tools.make_mp4(anim, file_name = video_dir / 'vagues.mp4')


In [ ]:
vmax = max([vague[i].max() for i in range(NbSim)] )
vmin = min([vague[i].min() for i in range(NbSim)] )
print(f"ampleur maximale des vagues {vmax:.1f} m")
print(f"creux maximal des vagues {vmin:.1f} m")